# 2. Attention analysis

Extract the decoder cross-attention for a compound-protein pair to see which residues
the model attends to, how that differs between the binding site and the rest of the
protein, and how a resistance mutation changes it. This is the analysis behind
Figure 4 of the paper, run here on the same input: the ABL1 kinase domain
(293 residues) and imatinib. Residue 93 of this construct is the gatekeeper T315.

**Runtime** — a few seconds on a laptop CPU.

## 1. Setup

The model runs on the CPU. `cfg.training.device` must be set **before** the model is
built, because the device is read at construction time.

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

warnings.filterwarnings('ignore')

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

CHECKPOINT = 'checkpoints/application.pth'
assert os.path.exists(CHECKPOINT), (
    f'{CHECKPOINT} not found. Download the weights from Zenodo first '
    '(see checkpoints/README.md).')

cfg = OmegaConf.load('configs/default.yaml')
cfg.model.type = 'aa3di_gmf'
cfg.training.device = 'cpu'
cfg.training.cuda = False

## 2. Build the model and load the weights

Attention maps need `get_attn_maps`, which `predict_single` does not expose, so the
model is built directly here.

In [ ]:
import torch
from omnibind.model import build_model

model = build_model(cfg)
checkpoint = torch.load(CHECKPOINT, map_location=cfg.training.device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(cfg.training.device)
model.eval()
print('model loaded, epoch', checkpoint['epoch'])

## 3. Residue-level attention profiles

`get_attn_maps` returns one map per decoder layer, of shape
`[batch, compound tokens, residues]`. Summing over the compound dimension gives one
weight per residue, as described in the paper. Each compound token spreads a total
weight of 1 over the residues, so a profile sums to the number of compound tokens and
its mean is fixed at `compound tokens / sequence length`. Weights are therefore
compared within one protein, not between proteins of different length.

In [ ]:
from rdkit import Chem

from omnibind.featurization import (
    mol_features_from_smiles, encode_aa_sequence, encode_3di_sequence,
)
from omnibind.predict import _pack_single

IMATINIB = 'CN1CCN(Cc2ccc(cc2)C(=O)Nc2ccc(C)c(Nc3nccc(n3)-c3cccnc3)c2)CC1'

# Binding site of the ABL1 kinase domain, in the numbering of this 293-residue
# construct. Residue 93 is the gatekeeper T315 of full-length ABL1.
BINDING_SITE = [26, 31, 34, 47, 49, 64, 68, 71, 77, 91, 93, 95, 96, 99,
                132, 137, 138, 139, 140, 148, 158, 159, 160]
GATEKEEPER = 93

proteins = pd.read_csv('data/sample/sample_proteins.csv').set_index('gene_name')


def attention_profiles(smiles, aa_sequence, sa_sequence):
    """Return one residue-level attention profile per decoder layer."""
    smiles = Chem.MolToSmiles(Chem.MolFromSmiles(smiles), canonical=True)
    atom_feature, adj = mol_features_from_smiles(smiles)
    packed = _pack_single([atom_feature], [adj],
                          [encode_aa_sequence(aa_sequence)],
                          [encode_3di_sequence(sa_sequence)])
    compounds, adjs, aas, sas, atom_num, aa_num, sa_num = packed

    device = cfg.training.device
    with torch.no_grad():
        *_, maps = model.get_attn_maps(
            compounds.to(device), adjs.to(device), aas.to(device), sas.to(device),
            atom_num, aa_num, sa_num)
    return [m.cpu().numpy()[0].sum(axis=0) for m in maps]


abl1 = proteins.loc['ABL1_KD']
profiles = attention_profiles(IMATINIB, abl1['aa_sequence'], abl1['3di_sequence'])

print('decoder layers:', len(profiles))
print('residues      :', len(profiles[0]))
print('gatekeeper    : residue', GATEKEEPER, '=', abl1['aa_sequence'][GATEKEEPER - 1])

## 4. Binding site attention across layers

Attention on the binding site residues, one column per decoder layer.

In [ ]:
site_matrix = np.array([[p[r - 1] for p in profiles] for r in BINDING_SITE])

fig, ax = plt.subplots(figsize=(5, 7))
im = ax.imshow(site_matrix, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(5))
ax.set_xticklabels(range(1, 6))
ax.set_yticks(range(len(BINDING_SITE)))
ax.set_yticklabels(BINDING_SITE, fontsize=8)
ax.set_xlabel('OmniBind Decoder layer')
ax.set_ylabel('Binding site residue')
ax.set_title('ABL1-imatinib\nBinding site attention weights across layers', fontsize=10)
fig.colorbar(im, ax=ax, label='Attention weight')
fig.tight_layout()

In [ ]:
peak_layer = int(np.argmax([p[GATEKEEPER - 1] for p in profiles]))
print(f'residue {GATEKEEPER} peaks in layer {peak_layer + 1} '
      f'with weight {profiles[peak_layer][GATEKEEPER - 1]:.3f}')

## 5. Binding site versus the rest of the protein

If the model has learnt where the compound binds, the binding site residues should
receive more attention than the rest.

In [ ]:
site_idx = [r - 1 for r in BINDING_SITE]
other_idx = [i for i in range(len(profiles[0])) if i not in site_idx]

site_mean = [p[site_idx].mean() for p in profiles]
other_mean = [p[other_idx].mean() for p in profiles]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(range(1, 6), site_mean, 'o-', color='tab:red', label='binding site')
ax.plot(range(1, 6), other_mean, 'o-', color='gray', label='non-binding')
ax.set_xlabel('OmniBind Decoder layer')
ax.set_ylabel('Mean attention weight')
ax.set_xticks(range(1, 6))
ax.legend()
fig.tight_layout()

pd.DataFrame({'layer': range(1, 6),
              'binding site': np.round(site_mean, 3),
              'non-binding': np.round(other_mean, 3)}).set_index('layer')

## 6. The T315I resistance mutation

T315I is the most common imatinib-resistance mutation. The mutant sequence differs
from the wild type at a single residue, so the change in the attention profile shows
how sensitive the model is to a single substitution.

In [ ]:
mutant = proteins.loc['ABL1_KD_T315I']

differences = [i + 1 for i, (w, m) in
               enumerate(zip(abl1['aa_sequence'], mutant['aa_sequence'])) if w != m]
print('amino acid differences:', differences)

mutant_profiles = attention_profiles(
    IMATINIB, mutant['aa_sequence'], mutant['3di_sequence'])

LAYER = peak_layer   # the layer where the gatekeeper peaks
delta = mutant_profiles[LAYER] - profiles[LAYER]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(np.arange(1, len(delta) + 1), delta,
       color=['tab:blue' if d < 0 else 'tab:red' for d in delta])
ax.axhline(0, color='black', lw=0.8)
ax.annotate('mutated', xy=(GATEKEEPER, delta[GATEKEEPER - 1]),
            xytext=(GATEKEEPER, delta.min() * 0.45),
            ha='center', arrowprops=dict(arrowstyle='->'))
ax.set_xlabel('Amino acid residue')
ax.set_ylabel('Weight change (Mutated - Wild-type)')
fig.tight_layout()

print(f'change at residue {GATEKEEPER}: {delta[GATEKEEPER - 1]:+.4f}')

## 7. Using your own data

Any compound-protein pair works, as long as the amino acid and 3Di sequences have
the same length. Binding site positions must be given in the numbering of the
sequence you supply.

The same extraction is available as a script, which saves the raw per-layer maps:

```bash
python scripts/attention_map.py \
    test.checkpoint_path=./checkpoints/application.pth \
    test.smiles="<compound_smiles>" \
    test.aa="<aa_sequence>" \
    test.sa="<3di_sequence>" \
    test.name="my_analysis"
```